# Automação de Extração do Relatório de Candidaturas da Gupy

- Extrair do site da Gupy as candidaturas
- Tratamento da base, padronização de nomes de colunas e formatos de dados
- Atualização do Power BI CANDIDATURAS GUPY

# Importando as Bibliotecas

In [ ]:
import datetime
import logging
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import time
import tkinter as tk
import win32com.client
from asyncio.log import logger
from contextlib import contextmanager
import datetime
from datetime import datetime as dt
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.datetime.today(), 0]

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Caminhos de Pastas

In [ ]:
CONFIG = {
    'id_processo': 5,
    'processos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CANDIDATURAS GUPY.xlsx'),
    'padrao': 'application_report_76663',
}

# Registra etapa do processo

In [ ]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

# Configurações Iniciais

In [ ]:
automacao = r'X:\Gestao_de_Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\AUTOMACAO'

chrome_options = Options()
chrome_options.add_argument(f"--user-data-dir={automacao}")
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-allow-origins=*")
chrome_options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get("https://afpesp.gupy.io/companies/")
    wait = WebDriverWait(driver, 150)
    actions = ActionChains(driver)
    
    print("⏳ Sincronizando a Gupy...")

except Exception as e:
    print(f"❌ Erro Geral: {e}")

finally:
    print(f"🏁 Acesso finalizado")

# Acessando a Gupy

In [ ]:
pyautogui.PAUSE = 1

# Área de login na Gupy
time.sleep(12)
pyautogui.click(x=125, y=525) # Clica no campo de login
time.sleep(2)
pyautogui.write("rodrigo.bernardes@afpesp.org.br")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(2)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.write("Spfc1935*")
pyautogui.click(x=253, y=630) # Acessar conta

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Acesso ao portal da Gupy com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

# Extrair relatório no portal da Gupy

In [ ]:
time.sleep(10)

# Home da Gupy
pyautogui.press("f5")
time.sleep(6)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(3)
pyautogui.click(x=549, y=492) # Candidaturas
time.sleep(2)
pyautogui.click(x=258, y=556) # XLSX
pyautogui.click(x=329, y=646) # Selecione um status de vaga
pyautogui.click(x=253, y=612) # Selecionar todos
pyautogui.click(x=1152, y=496) # Clicar fora
pyautogui.click(x=1115, y=632) # Selecione a vaga desejada
pyautogui.click(x=947, y=692) # Selecionar todas as vagas
pyautogui.click(x=1152, y=496) # Clicar fora

# Data atual = hoje - 1 dia
data_atual = datetime.date.today() - datetime.timedelta(days=1)

# Data início = data atual - 20 dias
data_inicio = data_atual - datetime.timedelta(days=20)

# Formato brasileiro: DD/MM/AAAA
fmt_br = "%d/%m/%Y"

data_atual_br = data_atual.strftime(fmt_br)
data_inicio_br = data_inicio.strftime(fmt_br)

time.sleep(1)
pyautogui.click(x=990, y=768) # Clica em Data fim
pyautogui.write(data_atual_br) # Digitar Data fim
pyautogui.click(x=334, y=759) # Clica em Data início
pyautogui.write(data_inicio_br) # Digitar Data início
pyautogui.click(x=1390, y=840) # Clica em Enviar por e-mail
time.sleep(2)
pyautogui.hotkey('alt', 'F4')

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Solicitação de download realizado com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

# Aguardando o recebimento do e-mail da Gupy com o arquivo para download

In [ ]:
root = tk.Tk()
root.withdraw()  # não mostra janela principal

while True:
    tecla = simpledialog.askstring("Continuar", "Digite S para continuar:")
    if (tecla or "").strip().upper() == "S":
        break

print(" ✅ Continuando o processo...")

# Carga e tratamento da base 

In [ ]:
import datetime

@contextmanager
def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def carregar_ou_criar_base(caminho: Path) -> pd.DataFrame:
    """Carrega base ou retorna DataFrame vazio."""
    if caminho.exists():
        return pd.read_excel(caminho, engine='openpyxl')
    logger.warning(f"⚠️ {caminho.name} não encontrado. Iniciando vazio.")
    return pd.DataFrame()

def registrar_etapa(caminho: Path, id_processo: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_processo, datetime.datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def filtrar_arquivos_novos(origem: Path, padrao: str = 'application_report_76663') -> pd.DataFrame:
    """Filtra e ordena arquivos novos."""
    arquivos = [f for f in origem.iterdir() if f.name.startswith(padrao)]
    
    if not arquivos:
        logger.warning("❌ Nenhum arquivo encontrado")
        return pd.DataFrame()
    
    df = pd.DataFrame([{'arquivo': f.name, 'caminho': f} for f in arquivos])
    df['data'] = pd.to_datetime(df['arquivo'].str[25:39], format='%Y%m%d%H%M%S')
    df = df.sort_values(by='data', ascending=True)
    
    logger.info(f"📂 {len(df)} arquivo(s) encontrado(s)")
    return df

def consolidar_incremental(base_atual: pd.DataFrame, base_nova: pd.DataFrame, chave: str) -> pd.DataFrame:
    """Consolidação incremental por chave única."""
    # Alinhar colunas
    todas_colunas = base_atual.columns.union(base_nova.columns)
    base_atual = base_atual.reindex(columns=todas_colunas)
    base_nova = base_nova.reindex(columns=todas_colunas)
    
    # Encontrar apenas novos registros
    novos = base_nova[~base_nova[chave].isin(base_atual[chave])]
    logger.debug(f"   +{len(novos)} novos registros")
    
    # Consolidar
    consolidada = pd.concat([base_atual, novos], ignore_index=True)
    consolidada = consolidada.drop_duplicates(subset=[chave], keep='first')
    
    return consolidada

def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path, sheet_name: str = 'CANDIDATURAS'):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name=sheet_name, index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName=sheet_name, ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")

# Main
if __name__ == "__main__":
    logger.info("=" * 80)
    logger.info("🔄 INICIANDO PROCESSAMENTO DE CANDIDATURAS")
    logger.info("=" * 80 + "\n")
    
    try:
        # Etapa 0: Registrar início
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
        # Etapa 1: Carregar base atual
        logger.info("📋 Carregando base atual...")
        candidaturas_atual = carregar_ou_criar_base(CONFIG['saida'])
        logger.info(f"✅ {len(candidaturas_atual)} registros na base atual\n")
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
        # Etapa 2: Filtrar arquivos
        logger.info("📂 Buscando arquivos novos...")
        df_arquivos = filtrar_arquivos_novos(CONFIG['origem'])
        
        if df_arquivos.empty:
            logger.warning("❌ Nenhum arquivo para processar")
            exit()
        
        logger.info("")
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
        # Etapa 3: Processar arquivos
        logger.info("🔄 Processando arquivos...\n")
        
        for idx, (_, row) in enumerate(df_arquivos.iterrows(), 1):
            nome_arquivo = row['arquivo']
            caminho_arquivo = row['caminho']
            
            logger.info(f"[{idx}/{len(df_arquivos)}] {nome_arquivo}")
            
            try:
                # Carregar
                candidaturas_novas = pd.read_excel(caminho_arquivo)
                logger.debug(f"   ✓ {len(candidaturas_novas)} registros carregados")
                
                # Consolidar
                candidaturas_atual = consolidar_incremental(
                    candidaturas_atual,
                    candidaturas_novas,
                    chave='ID da aplicação'
                )
                
                # Mover arquivo
                destino = CONFIG['movidos'] / nome_arquivo
                shutil.move(str(caminho_arquivo), str(destino))
                logger.info(f"   ✅ Movido para arquivos processados\n")
                
            except Exception as e:
                logger.error(f"   ❌ Erro: {e}\n")
                continue
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 3)
        
        # Etapa 4: Salvar resultado final
        logger.info("💾 Salvando resultado final...")
        criar_excel_com_tabela(candidaturas_atual, CONFIG['saida'])
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 4)
        
        logger.info("\n" + "=" * 80)
        logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
        logger.info(f"   Total de registros: {len(candidaturas_atual)}")
        logger.info("=" * 80)
        
    except Exception as e:
        logger.error(f"\n❌ ERRO CRÍTICO: {e}")
        import traceback
        logger.error(traceback.format_exc())

# Atualização do Power BI - CANDIDATURAS GUPY

In [ ]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestao_de_Pessoas\Analytics\09 - Power BI\CANDIDATURAS GUPY.pbix"
os.startfile(path_pbi) # Abre o arquivo pelo Windows
time.sleep(30)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(40)
pyautogui.click(x=1294, y=109) # Publicar
time.sleep(5)
pyautogui.click(x=873, y=476) # Salvar
time.sleep(7)
pyautogui.click(x=712, y=379) # Clicar em GESTÃO DE PESSOAS
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(5)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Resumo de Finalização do Processo

In [ ]:
print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')